# Plot Profiles from ICON 3D Data

Here, we proceed with ICON data.  We will have a look at 3D ICON data.

## Load Python Libraries 

In [ ]:
%matplotlib inline

# system libs
import os, sys, glob

# array operators and netcdf datasets
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

# plotting
import pylab as plt
import seaborn as sns
sns.set_context('talk')


## Open Data with `xarray`

### General Paths

In [ ]:
exercise_path = '/home/b/b383413/workspace'
data_path = f'{exercise_path}/data'
icon_data_path = f'{data_path}/icon'

This is very similar to the tasks already done.

In [ ]:
os.listdir(icon_data_path)

Now, we select the `experiment`folder in which pre-calculated experiments are stored. **Later, you need to change this directory to your ICON output directory!** 

In [ ]:
#exp_name = 'icon_lam_1dom_lpz-base'
#exp_path = f'{icon_data_path}/experiments/{exp_name}'
exp_name = 'cesar1-20240806-exp001testothertime'
exp_path = f'/home/b/b383413/workspace/icon-build/experiments/{exp_name}'

In [ ]:
sorted( os.listdir( exp_path, )) 

Ui, many files! Again, files with `.nc` are our netcdf data. The suffix `_mean` indicates files for domain-mean quantities that have been calculated with `cdo`.


Let's have a look at the different categories:

In [ ]:
#sorted( glob.glob( f'{exp_path}/*DOM01_ML_20210516T120000Z.nc') )
sorted( glob.glob( f'{exp_path}/*DOM01_ML_20240806T060000Z.nc') )

* `2d_rad` : variables related to radiation
* `2d_cloud` : variable related to clouds & precipitation
* `2d_surface` : variable related to surface properties or surface budgets
* `3d_full_base` : main prognostic variables
* `3d_full_qmix` : hydrometeor mass mixing ratios
* `3d_full_qnum` : hydrometeor number mixing ratios
* `3d_full_tend` : tendencies of some prognostic variables (incl. physic parameterizations)
* `3d_full_aux` : some additional 3d fields

**Please, open a terminal and look at the file content.** (Using either `cdo sinfov <filename>` or `ncdump -h <filename>`).


### Base Files 

In [ ]:
#set = xr.open_dataset( f'{exp_path}/3d_full_base_DOM01_ML_20210516T120000Z.nc', chunks={} ) 
dset = xr.open_dataset( f'{exp_path}/3d_full_base_DOM01_ML_20240806T060000Z.nc', chunks={} ) 

This is the content of the dataset. 

In [ ]:
dset.data_vars

It is easy to guess for which each variable name stands. If you don't like to guess, you could further look at dataset attributes: 

In [ ]:
dset['u']

### Calculate Average Profiles

In [ ]:
dset_mean = dset.mean( 'ncells' )

There is an additional time dimension of `length = 1`. This is not needed and we get rid of it if the `queeze`method:

In [ ]:
dset_mean = dset_mean.squeeze()

## Plotting the Profiles 

### Model Level as Vertical Coordinate 

The coordinate `height` denotes the ICON model level:

In [ ]:
dset['height']

These are terrain-following coordinates and additional data are needed to relate model level to a geometric height above ground or perhaps to atmospheric pressure.


Let' shave a look at the pressure profile first to understand how model level are sorted.

In [ ]:
plt.figure( figsize = (4,6))
dset_mean['pres'].plot( y = 'height', lw = 3)
plt.title( 'Pressure / Pa', fontweight = 'bold')
sns.despine()

OK; pressure is lowest at model level equal to one, i.e. model levels increase from top to bottom!

### Average Pressure on Vertical Axis

Atmospheric scientist like the pressure coordinate. Here, we do the trick that we still plot model level, but show the typical average pressure which each level corresponds to. This practice is very often sufficient. 

We prepare the average pressure:

In [ ]:
p_average = dset_mean['pres'] / 1e2   # conversion into hPa
p_average.attrs['units'] = 'hPa'

and then assign the pressure to the `height`coordinate:

In [ ]:
dset_mean = dset_mean.assign_coords( {'height': p_average})
dset_mean = dset_mean.rename( {'height': 'pressure'})

dset_mean = dset_mean.sel( pressure = slice(200,1000))

finally, we plot the profiles of the atmopsheric variables as multi-panel plot:

In [ ]:
figure, axs = plt.subplots( ncols = 3, nrows = 2, figsize = (14,16),)
axs = axs.flatten()
plt.subplots_adjust( hspace = 0.5, wspace = 0.5)
for i, varname in enumerate( ['u', 'v', 'qv', 'clc', 'temp', 'theta_v'] ):
    
    plt.sca( axs[i] )
    
    v = dset_mean[varname]
    v.plot( y = 'pressure', lw = 3 )
    
    plt.title('%s / %s' % (v.long_name, v.units), pad = 20, fontweight = 'bold')
    plt.ylim(1000., 200.)
    sns.despine()

## Tasks

Use this notebook to explore the other `3d_*nc` files:

* Do you understand the meaning of the different variables?

* Specifically, how are liquid and frozen hydrometeors distributed?